# Finding depth and breadth of conditionality in SPDX

### 1. Setting up

Conditions: Manually extracted list of all conditions in SPDX 3.0

Schema: My manually extracted schema from the SPDX specifications. I chose the manually extracted schema over the official schema at this stage in order to be consistent with my other work, which had findings based off of my own schema.

In [11]:
import json

CLASSES_FILE = 'classes_core.txt'
SCHEMA_FILE = 'schema.json'
OUTPUT_FILE = "classes_depth.csv"


### 2. Loading classes and schema

In [12]:
with open(CLASSES_FILE, 'r') as f:
    classes = {line.strip() for line in f}

print(f"{len(classes)}  classes loaded: {classes}\n")

with open(SCHEMA_FILE, 'r') as f:
    schema = json.load(f)

20  classes loaded: {'Relationship', 'Artifact', 'Agent', 'PositiveIntegerRange', 'Organization', 'PackageVerificationCode', 'Bom', 'Bundle', 'ExternalIdentifier', 'ExternalMap', 'DictionaryEntry', 'Annotation', 'LifecycleScopedRelationship', 'CreationInfo', 'ElementCollection', 'Element', 'IndividualElement', 'Hash', 'NamespaceMap', 'Person'}



### 3. Finding all conditional classes

In [13]:
ancestor_graph = {}
all_nodes = set(schema["$defs"].keys())

for parent_name, definition in schema["$defs"].items():
    def find_refs_in_definition(obj):
        refs = set()
        if isinstance(obj, dict):
            for key, value in obj.items():
                if key == "$ref" and isinstance(value, str):
                    child_name = value.split('/')[-1]
                    if child_name in all_nodes:
                        refs.add(child_name)
                else:
                    refs.update(find_refs_in_definition(value))
        elif isinstance(obj, list):
            for item in obj:
                refs.update(find_refs_in_definition(item))
        return refs

    children = find_refs_in_definition(definition)
    for child in children:
        if child not in ancestor_graph:
            ancestor_graph[child] = set()
        ancestor_graph[child].add(parent_name)

final_results = {}
for target_class in classes:
    if target_class not in all_nodes:
        final_results[target_class] = {'path': [], 'length': 0, 'error': 'Class not found in schema definitions.'}
        continue

    all_paths = []

    def find_paths_recursive(current_node, current_path):
        if current_node in current_path:
            return

        new_path = [current_node] + current_path

        if current_node not in ancestor_graph:
            all_paths.append(new_path)
            return

        has_parents = False
        for parent in ancestor_graph[current_node]:
            has_parents = True
            find_paths_recursive(parent, new_path)
        
        if not has_parents:
            all_paths.append(new_path)

    find_paths_recursive(target_class, [])

    if not all_paths:
        longest_path = [target_class]
    else:
        longest_path = max(all_paths, key=len)

    final_results[target_class] = {
        'path': longest_path,
        'length': len(longest_path)
    }


### 4. Printing results

In [14]:
print("--- Furthest Ancestor Analysis ---")
for class_name, result in final_results.items():
    print(f"\nClass: '{class_name}'")
    if 'error' in result:
        print(f"  Error: {result['error']}")
        continue

    path = " -> ".join(result['path'])
    length = result['length']
    
    print(f"  Length of Longest Path: {length}")
    print(f"  Path: {path}")

--- Furthest Ancestor Analysis ---

Class: 'Relationship'
  Length of Longest Path: 4
  Path: security_VexUnderInvestigationVulnAssessmentRelationship -> security_VexVulnAssessmentRelationship -> security_VulnAssessmentRelationship -> Relationship

Class: 'Artifact'
  Length of Longest Path: 5
  Path: security_VexUnderInvestigationVulnAssessmentRelationship -> security_VexVulnAssessmentRelationship -> security_VulnAssessmentRelationship -> software_SoftwareArtifact -> Artifact

Class: 'Agent'
  Length of Longest Path: 8
  Path: security_VexUnderInvestigationVulnAssessmentRelationship -> security_VexVulnAssessmentRelationship -> security_VulnAssessmentRelationship -> software_SoftwareArtifact -> Artifact -> Element -> CreationInfo -> Agent

Class: 'PositiveIntegerRange'
  Length of Longest Path: 2
  Path: software_Snippet -> PositiveIntegerRange

Class: 'Organization'
  Length of Longest Path: 2
  Path: SpdxOrganization -> Organization

Class: 'PackageVerificationCode'
  Length of Longe

### 5. Saving results

In [16]:
import csv

with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as csvfile:
    # Create a writer object
    writer = csv.writer(csvfile)

    # Write the header row
    writer.writerow(['class_name', 'max_depth', 'path'])

    # Loop through your dictionary and write data
    for class_name, result in final_results.items():
        if 'error' in result:
            # Write a row for an error case
            writer.writerow([class_name, 'ERROR', result['error']])
        else:
            # Format the path and write a row for a success case
            path_string = " -> ".join(result['path'])
            writer.writerow([class_name, result['length'], path_string])
